# 06 · ephys ramp: rheobase, up/down firing, hysteresis, Rin, rebound

This notebook analyses the **ramp recordings** (A4 / M4). In the example data each sweep has:
a 500 ms ramp **0 → +100 pA**, straight back **+100 → 0 pA** (500 ms), 2 s at 0, a 500 ms ramp **0 → −100 pA**,
an **abrupt return to 0 pA**, then 15 s at 0 (rebound). The segments are read from each file's command waveform.

Input: `02_ephys_anomaly-fix/output/analysis_ready.csv`. The injected current is the **command** (plus any bias).
Settings: `config.yaml` in this folder. Method: `README.md`.

| Step | What happens | Review |
|---|---|---|
| 1 | select the ready ramp recordings | which files |
| 2 | **detect spikes and attenuated events in every sweep and save a detection figure (PNG + SVG)** | open the figures |
| 3 | per-sweep and per-recording features, from the same spikes | first-spike current, hysteresis, Rin, rebound |
| 4 | save | |

## Setup
`OVERWRITE = True` replaces existing output files. Close them in Excel first.

In [ ]:
OVERWRITE = True

from pathlib import Path

import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)

from ephys_analysis.ramp import RampSettings, detect_all, feature_table
from ephys_analysis.spikes import SpikeSettings
from ephys_analysis.utils import RunLog, check_writable, find_repo_root, load_config, repo_path, stamp, write_csv

REPO = find_repo_root()
cfg = load_config(REPO / "06_ephys_ramp")
sp = SpikeSettings.from_config(cfg["spikes"])
st = RampSettings.from_config(cfg["ramp"])
out = REPO / "06_ephys_ramp" / "output"
figures = out / "figures"
targets = [out / "spikes.csv", out / "attenuated_events.csv", out / "detection.csv",
           out / "ramp_sweeps.csv", out / "ramp_features.csv", out / "ap_waveforms.csv"]
check_writable(targets, OVERWRITE)
log = RunLog(cfg, out)
print(st)

## Step 1 · Select the ramp recordings

In [ ]:
ready = pd.read_csv(repo_path(cfg, cfg["recordings_table"]), dtype={"cell_tag": "string"})
selection = ready[ready["family"].isin(cfg["families"])].reset_index(drop=True)
display(selection[["recording_id", "cell_tag", "abf_filename", "voltage_source", "current_source", "bias_current_pA", "corrections"]])

## Step 2 · Detect spikes and save a detection figure for every file
Spikes are detected over the whole sweep with the same detector as 04 and 05 and labelled by **period**:
`pre`, `up` (rising ramp), `down` (falling ramp), `mid`, `neg` (negative ramp), `post` (after the abrupt return to
0 pA: rebound). **Attenuated events** (○; peaks that are not spikes, e.g. depolarisation block) are listed separately.

Figure per recording (`output/figures/<recording_id>.png` and `.svg`): all sweeps overlaid over the ramps with the
command below, whole sweeps stacked (rebound), and every spike aligned to its threshold coloured by period.

In [ ]:
spikes, events, detection = detect_all(selection, sp, st, log, figures, tuple(cfg["figure_formats"]))
display(detection)

In [ ]:
# Review: every detection figure. If detection is wrong, change `spikes:` in config.yaml and re-run Step 2.
for f in detection["figure"].dropna():
    if f:
        print(Path(f).name)
        display(Image(filename=f, width=1100))

**Checkpoint.** Only continue when the detection is right: the features below are computed from exactly
these spikes.

## Step 3 · Features
**Per sweep** (`ramp_sweeps.csv`):
- **current at the first spike** on the up-ramp, its threshold and latency; the **rheobase** is its mean over the sweeps with no spike in the 0.5 s before the ramp;
- spikes on the up- and down-ramp, highest instantaneous rate on the up-ramp;
- **hysteresis** = current at the first up-ramp spike − current at the last down-ramp spike (> 0: firing continues to
  lower currents on the way down; < 0: it stops earlier);
- **Rin** from the negative ramp (slope of V against I, spikes removed, first 50 ms skipped);
- **rebound** after the abrupt return to 0 pA (peak, spikes within 5 s and to the end, latency, bursts).

**Per recording** (`ramp_features.csv`): the **median over the sweeps** (repeats of the same protocol within one
recording) and the AP waveform of the first valid up-ramp spike.

In [ ]:
sweeps, features, waves = feature_table(selection, spikes, events, st, log)
display(sweeps.round(2))

In [ ]:
display(features.T)

## Step 4 · Save

In [ ]:
for df, p in zip((spikes, events, detection, sweeps, features, waves), targets):
    write_csv(stamp(df, log.run_id), p)
    print("written:", p)
log.finish(targets)